In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mc
from scipy import stats
import statsmodels.stats.multitest as multitest
import warnings
import os

warnings.filterwarnings('ignore')

# =============================================================================
# Settings
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

def clean_and_convert_to_nan(vals):
    s_vals = pd.Series(vals).astype(str).str.strip()
    s_vals = s_vals.str.replace(r'\.E\+', 'E+', regex=True)
    s_vals = s_vals.str.replace(r'\.E\-', 'E-', regex=True)
    s_vals = s_vals.replace(['Undetermined', '-', 'nan', 'NaN', '#VALUE!', '', 'None'], np.nan)
    return pd.to_numeric(s_vals, errors='coerce')

def lighten_hex(hex_color, factor=0.3):
    r, g, b = mc.to_rgb(hex_color)
    return mc.to_hex((r + (1-r)*factor, g + (1-g)*factor, b + (1-b)*factor))

inulin_color = lighten_hex('#800080', 0.3)

taxa_files = {
    'Bifidobacterium': 'Bifidobacterium(qPCR).csv',
    'Faecalibacterium': 'Faecalibacterium(qPCR).csv',
    'Blautia': 'Blautia(qPCR).csv',
    'Butyricicoccus': 'Butyricicoccus(qPCR).csv',
    'Anaerostipes caccae': 'A.caccae.csv'
}
order_4f = ['Bifidobacterium', 'Faecalibacterium', 'Blautia', 'Butyricicoccus', 'Anaerostipes caccae']

def get_valid_delta(df, sub_name):
    donor_cols = [c for c in df.columns if c.startswith('HS-')]
    ctrl = clean_and_convert_to_nan(df[df['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
    sub = clean_and_convert_to_nan(df[df['KULFFI'].str.strip() == sub_name][donor_cols].iloc[0])
    return np.log10(sub) - np.log10(ctrl)

# =============================================================================
# Data Processing
# =============================================================================
plot_data_4f = []
p_vals_4f = []
valid_taxa_4f = []

for taxon in order_4f:
    if not os.path.exists(taxa_files[taxon]): continue
    try: df_tax = pd.read_csv(taxa_files[taxon])
    except UnicodeDecodeError: df_tax = pd.read_csv(taxa_files[taxon], encoding='shift_jis')

    delta_tax = get_valid_delta(df_tax, 'Inulin')
    valid_delta = delta_tax.dropna()

    if len(valid_delta) > 0:
        c_vals = clean_and_convert_to_nan(df_tax[df_tax['KULFFI'].str.strip() == 'Control'][[c for c in df_tax.columns if c.startswith('HS-')]].iloc[0]).loc[valid_delta.index]
        t_vals = clean_and_convert_to_nan(df_tax[df_tax['KULFFI'].str.strip() == 'Inulin'][[c for c in df_tax.columns if c.startswith('HS-')]].iloc[0]).loc[valid_delta.index]
        try: stat, p = stats.wilcoxon(t_vals, c_vals)
        except ValueError: p = 1.0

        p_vals_4f.append(p)
        valid_taxa_4f.append(taxon)
        for d in valid_delta: plot_data_4f.append({'Taxon': taxon, 'Delta': d})

if len(p_vals_4f) > 0:
    _, q_vals_4f, _, _ = multitest.multipletests(p_vals_4f, alpha=0.05, method='fdr_bh')
    q_dict_4f = dict(zip(valid_taxa_4f, q_vals_4f))
else:
    q_dict_4f = {}

df_4f = pd.DataFrame(plot_data_4f)
df_4f['Taxon'] = pd.Categorical(df_4f['Taxon'], categories=order_4f, ordered=True)

# =============================================================================
# Plotting
# =============================================================================
# Width is 4.5 for a single hue group (Inulin only)
fig, ax = plt.subplots(figsize=(4.5, 6))

sns.boxplot(x='Taxon', y='Delta', data=df_4f, palette=[inulin_color]*len(order_4f), width=0.5, showfliers=False, ax=ax)
sns.stripplot(x='Taxon', y='Delta', data=df_4f, color='black', alpha=0.5, size=4, jitter=0.2, ax=ax)

for patch in ax.patches:
    patch.set_edgecolor('black')
    patch.set_linewidth(1.5)
for line in ax.lines:
    line.set_color('black')
    line.set_linewidth(1.5)

ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_ylabel(r'$\Delta$ Abundance' + '\n' + r'(Log$_{\mathbf{10}}$ copies/mL)', fontsize=14, fontweight='bold', labelpad=10)
ax.set_xlabel('')
ax.set_xticklabels([''] * len(order_4f))

# Dynamic Y limits to ensure no artifacts
y_min, y_max = df_4f['Delta'].min(), df_4f['Delta'].max()
y_range = y_max - y_min
ax.set_ylim(y_min - (y_range * 0.05), y_max + (y_range * 0.15))

ax.tick_params(axis='y', labelsize=12, width=1.5, length=5)
ax.tick_params(axis='x', width=1.5, length=5)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
for s in ['left', 'bottom']: ax.spines[s].set_linewidth(1.5)

# Significance Stars
for i, taxon in enumerate(order_4f):
    if taxon in q_dict_4f:
        q_val = q_dict_4f[taxon]
        star = '***' if q_val < 0.001 else '**' if q_val < 0.01 else '*' if q_val < 0.05 else ''
        if star:
            g_max = df_4f[df_4f['Taxon'] == taxon]['Delta'].max()
            ax.text(i, g_max + (y_range * 0.02), star, ha='center', va='bottom', fontsize=14, fontweight='bold', color='#008B8B')

# X-axis Labels
y_base = ax.get_ylim()[0] - (y_range * 0.03)
for i, taxon in enumerate(order_4f):
    if taxon == 'Anaerostipes caccae':
        ax.text(i, y_base, 'Anaerostipes caccae', rotation=45, ha='right', va='top', fontstyle='italic', fontweight='bold', fontsize=12)
    else:
        ax.text(i, y_base, taxon + ' ', rotation=45, ha='right', va='top', fontstyle='italic', fontweight='bold', fontsize=12)
        ax.text(i + 0.16, y_base - (y_range * 0.02), 'spp.', rotation=45, ha='right', va='top', fontstyle='normal', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('Figure_4f.pdf', dpi=DPI_SETTING, bbox_inches='tight', transparent=True)
plt.close()